<div style="background: linear-gradient(135deg, #1a3a5c 0%, #0d6efd 100%); padding: 40px 30px; border-radius: 16px; text-align: center; margin-bottom: 20px;">
  <h1 style="color: white; font-size: 2.2em; margin: 0; font-family: 'Georgia', serif;">🏥 Detección de Fraude en Seguros Médicos</h1>
  <h2 style="color: #aed6f1; font-size: 1.4em; margin: 10px 0 0;">Usando Autoencoders como modelo de Anomaly Detection</h2>
  <p style="color: #d6eaf8; margin: 15px 0 0; font-size: 1.05em;">Programa Aplicado de Machine Learning · Doctoral Student Gladys Choque Ulloa</p>
</div>


---
## 📋 1. Contexto del Estudio

### ¿Qué es el fraude en seguros médicos?

El fraude en el sistema de salud ocurre cuando proveedores médicos, pacientes o terceros envían **reclamaciones falsas o infladas** al sistema de seguros con el fin de obtener pagos indebidos. Los patrones más comunes incluyen:

| Tipo de fraude | Descripción |
|---|---|
| **Upcoding** | Cobrar por procedimientos más costosos de los realizados |
| **Phantom billing** | Facturar servicios que nunca se prestaron |
| **Unbundling** | Separar servicios que deben facturarse juntos |
| **Duplicate billing** | Enviar la misma reclamación múltiples veces |
| **Kickbacks** | Pagos ilegales entre proveedores y laboratorios |

### Magnitud del problema
- 🔴 El fraude representa entre el **3% y 10%** del gasto total en salud a nivel mundial
- 💰 En EE.UU. se estiman pérdidas de más de **\$100 mil millones** anuales
- 🌎 En Latinoamérica el problema es igualmente crítico pero menos estudiado
- ⚠️ Los modelos supervisados clásicos **fallan** porque el fraude evoluciona constantemente

### ¿Por qué usar un Autoencoder?

Un autoencoder es ideal para este problema porque:

> **El modelo aprende el patrón de las reclamaciones NORMALES. Cuando se enfrenta a una reclamación fraudulenta — que tiene un patrón diferente — no puede reconstruirla bien, generando un ERROR ALTO.**

```
Reclamación Normal  → Autoencoder → Error bajo  ✅  → LEGÍTIMA
Reclamación Fraude  → Autoencoder → Error alto  🚨  → SOSPECHOSA
```

Esta es una técnica de **detección de anomalías no supervisada**: no necesitamos las etiquetas de fraude para entrenar, solo para evaluar.


---
## 🎯 2. Problemática a Solucionar

**Dataset:** Healthcare Fraud Detection Dataset  
**Registros:** 10,000 reclamaciones médicas  
**Fraude real:** ~8.3% de los casos (829 fraudes)

### Desafíos principales:

1. **Desbalance severo de clases**: 91.7% legítimas vs 8.3% fraudulentas
2. **Fraude camuflado**: Las reclamaciones fraudulentas imitan patrones legítimos
3. **Sin historial suficiente**: Nuevos proveedores pueden no tener antecedentes
4. **Variabilidad natural**: No todo reclamo alto es fraude

### Objetivo del notebook:
> Construir un **Autoencoder** que, entrenado únicamente con reclamaciones legítimas, sea capaz de detectar reclamaciones fraudulentas a través del error de reconstrucción, y visualizar el proceso de forma clara y didáctica.


---
## ⚙️ 3. Instalación de Librerías

In [ ]:
# Ejecutar solo si es necesario
!pip install tensorflow scikit-learn pandas numpy matplotlib seaborn plotly -q

---
## 📦 4. Importación de Librerías

In [ ]:
# ── Core ────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Visualización ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Machine Learning ─────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, precision_recall_curve,
                              average_precision_score, f1_score)

# ── Deep Learning ────────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

# ── Configuración estética global ────────────────────────────────────────────
PALETTE = {
    'fraud'  : '#e74c3c',   # rojo
    'legit'  : '#2ecc71',   # verde
    'primary': '#2980b9',   # azul
    'dark'   : '#1a252f',   # fondo oscuro
    'accent' : '#f39c12',   # amarillo/naranja
    'purple' : '#8e44ad',
}

plt.rcParams.update({
    'figure.facecolor'  : '#1a252f',
    'axes.facecolor'    : '#212f3d',
    'axes.edgecolor'    : '#34495e',
    'axes.labelcolor'   : '#ecf0f1',
    'axes.titlecolor'   : '#ecf0f1',
    'xtick.color'       : '#bdc3c7',
    'ytick.color'       : '#bdc3c7',
    'text.color'        : '#ecf0f1',
    'grid.color'        : '#2c3e50',
    'grid.linestyle'    : '--',
    'grid.alpha'        : 0.5,
    'font.family'       : 'DejaVu Sans',
    'axes.titlesize'    : 14,
    'axes.labelsize'    : 12,
})

np.random.seed(42)
tf.random.set_seed(42)

print('✅ Librerías importadas correctamente')
print(f'   TensorFlow: {tf.__version__}')
print(f'   Pandas:     {pd.__version__}')

---
## 📂 5. Carga del Dataset

In [ ]:
# ── Cargar datos ─────────────────────────────────────────────────────────────
# Si estás en Google Colab, sube el archivo con:
#   from google.colab import files
#   files.upload()
# Luego cambia la ruta a: 'healthcare_fraud_detection.csv'

df = pd.read_csv('healthcare_fraud_detection.csv')

print('=' * 55)
print('       RESUMEN DEL DATASET')
print('=' * 55)
print(f'  Registros totales : {len(df):,}')
print(f'  Variables         : {df.shape[1]}')
print(f'  Fraudes           : {df["Is_Fraud"].sum():,} ({df["Is_Fraud"].mean()*100:.1f}%)')
print(f'  Legítimas         : {(df["Is_Fraud"]==0).sum():,} ({(df["Is_Fraud"]==0).mean()*100:.1f}%)')
print(f'  Valores faltantes : {df.isnull().sum().sum()}')
print('=' * 55)

df.head(3)

---
## 📊 6. Análisis Exploratorio (EDA)

Antes de construir el modelo, necesitamos **entender los datos** y verificar si existen diferencias observables entre reclamaciones legítimas y fraudulentas.

In [ ]:
# ── Gráfico 1: Distribución de fraude + variables clave ─────────────────────
fig = plt.figure(figsize=(18, 13))
fig.suptitle('Análisis Exploratorio — Healthcare Fraud Detection',
             fontsize=17, fontweight='bold', color='white', y=0.98)

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.55, wspace=0.4)

fraud_mask = df['Is_Fraud'] == 1

# --- Panel 1: Donut chart fraude vs legítimo ---
ax0 = fig.add_subplot(gs[0, 0])
sizes  = [df['Is_Fraud'].sum(), (df['Is_Fraud'] == 0).sum()]
labels = ['Fraude\n8.3%', 'Legítimo\n91.7%']
colors_pie = [PALETTE['fraud'], PALETTE['legit']]
wedges, texts = ax0.pie(sizes, labels=labels, colors=colors_pie,
                         startangle=90, wedgeprops=dict(width=0.55),
                         textprops={'fontsize': 11, 'color': 'white'})
ax0.set_title('Distribución de Clases', pad=10)

# --- Panel 2: Claim Amount por clase ---
ax1 = fig.add_subplot(gs[0, 1])
ax1.hist(df[~fraud_mask]['Claim_Amount'], bins=50, alpha=0.75,
         color=PALETTE['legit'], label='Legítimo', density=True)
ax1.hist(df[fraud_mask]['Claim_Amount'], bins=50, alpha=0.75,
         color=PALETTE['fraud'], label='Fraude', density=True)
ax1.set_title('Monto Reclamado ($)')
ax1.set_xlabel('Claim Amount')
ax1.set_ylabel('Densidad')
ax1.legend(fontsize=9)
ax1.axvline(df[fraud_mask]['Claim_Amount'].mean(), color=PALETTE['fraud'],
            linestyle='--', linewidth=1.5,
            label=f'Media fraude: ${df[fraud_mask]["Claim_Amount"].mean():.0f}')
ax1.axvline(df[~fraud_mask]['Claim_Amount'].mean(), color=PALETTE['legit'],
            linestyle='--', linewidth=1.5)
ax1.legend(fontsize=8)

# --- Panel 3: Days between service and claim ---
ax2 = fig.add_subplot(gs[0, 2])
ax2.hist(df[~fraud_mask]['Days_Between_Service_and_Claim'], bins=35,
         alpha=0.75, color=PALETTE['legit'], label='Legítimo', density=True)
ax2.hist(df[fraud_mask]['Days_Between_Service_and_Claim'], bins=35,
         alpha=0.75, color=PALETTE['fraud'], label='Fraude', density=True)
ax2.set_title('Días entre Servicio y Reclamo')
ax2.set_xlabel('Días')
ax2.set_ylabel('Densidad')
ax2.legend(fontsize=9)

# --- Panel 4: Tasa de fraude por especialidad ---
ax3 = fig.add_subplot(gs[1, :])
fraud_by_spec = df.groupby('Provider_Specialty').agg(
    fraud_rate=('Is_Fraud', 'mean'),
    total=('Is_Fraud', 'count')
).reset_index().sort_values('fraud_rate', ascending=True)

colors_bar = [PALETTE['fraud'] if r > 0.085 else PALETTE['primary']
              for r in fraud_by_spec['fraud_rate']]
bars = ax3.barh(fraud_by_spec['Provider_Specialty'],
                fraud_by_spec['fraud_rate'] * 100,
                color=colors_bar, edgecolor='none', height=0.55)
ax3.axvline(df['Is_Fraud'].mean() * 100, color=PALETTE['accent'],
            linestyle='--', linewidth=2, label=f'Media global ({df["Is_Fraud"].mean()*100:.1f}%)')
for bar, val in zip(bars, fraud_by_spec['fraud_rate']):
    ax3.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
             f'{val*100:.1f}%', va='center', fontsize=10, color='white')
ax3.set_title('Tasa de Fraude por Especialidad Médica')
ax3.set_xlabel('Tasa de Fraude (%)')
ax3.legend(fontsize=9)
ax3.set_xlim(0, 14)

# --- Panel 5: Fraude por tipo de visita ---
ax4 = fig.add_subplot(gs[2, 0])
fraud_visit = df.groupby('Visit_Type')['Is_Fraud'].mean() * 100
ax4.bar(fraud_visit.index, fraud_visit.values,
        color=[PALETTE['primary'], PALETTE['purple'], PALETTE['accent']],
        edgecolor='none')
ax4.set_title('Fraude por Tipo de Visita')
ax4.set_ylabel('Tasa de Fraude (%)')
for i, v in enumerate(fraud_visit.values):
    ax4.text(i, v + 0.1, f'{v:.1f}%', ha='center', fontsize=10, color='white')

# --- Panel 6: Fraude por tipo de seguro ---
ax5 = fig.add_subplot(gs[2, 1])
fraud_ins = df.dropna(subset=['Insurance_Type']).groupby('Insurance_Type')['Is_Fraud'].mean() * 100
ax5.bar(fraud_ins.index, fraud_ins.values,
        color=[PALETTE['legit'], PALETTE['primary'], PALETTE['accent'], PALETTE['fraud']],
        edgecolor='none')
ax5.set_title('Fraude por Tipo de Seguro')
ax5.set_ylabel('Tasa de Fraude (%)')
ax5.tick_params(axis='x', labelrotation=15)
for i, v in enumerate(fraud_ins.values):
    ax5.text(i, v + 0.1, f'{v:.1f}%', ha='center', fontsize=9, color='white')

# --- Panel 7: Age distribution ---
ax6 = fig.add_subplot(gs[2, 2])
ax6.hist(df[~fraud_mask]['Patient_Age'], bins=30, alpha=0.75,
         color=PALETTE['legit'], label='Legítimo', density=True)
ax6.hist(df[fraud_mask]['Patient_Age'], bins=30, alpha=0.75,
         color=PALETTE['fraud'], label='Fraude', density=True)
ax6.set_title('Edad del Paciente')
ax6.set_xlabel('Edad')
ax6.set_ylabel('Densidad')
ax6.legend(fontsize=9)

plt.savefig('eda_overview.png', dpi=130, bbox_inches='tight',
            facecolor='#1a252f')
plt.show()
print('📊 EDA guardado como eda_overview.png')

In [ ]:
# ── Gráfico 2: Correlación con fraude ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Comparación Numérica: Fraude vs Legítimo', fontsize=15,
             fontweight='bold', color='white')

num_cols = ['Claim_Amount', 'Approved_Amount',
            'Days_Between_Service_and_Claim',
            'Number_of_Claims_Per_Provider_Monthly',
            'Patient_Age', 'Prior_Visits_12m']

means_fraud = df[df['Is_Fraud'] == 1][num_cols].mean()
means_legit = df[df['Is_Fraud'] == 0][num_cols].mean()

# Normalizar para comparar en escala relativa
norm_fraud = means_fraud / means_legit

labels_short = ['Claim\nAmount', 'Approved\nAmount', 'Días\nServicio',
                'Claims\nMensuales', 'Edad\nPaciente', 'Visitas\n12m']
colors_radar = [PALETTE['fraud'] if v > 1.05 else
                (PALETTE['accent'] if v > 0.97 else PALETTE['legit'])
                for v in norm_fraud.values]

bars = axes[0].bar(labels_short, norm_fraud.values, color=colors_radar, edgecolor='none')
axes[0].axhline(1.0, color='white', linestyle='--', linewidth=1.5,
                label='Igual que legítimo')
axes[0].set_title('Ratio Fraude/Legítimo por variable\n(>1 = fraude tiene valor mayor)')
axes[0].set_ylabel('Ratio')
axes[0].legend(fontsize=9)
for bar, val in zip(bars, norm_fraud.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.2f}x', ha='center', fontsize=10, color='white')

# Heatmap de correlación numérica
corr_df = df[num_cols + ['Is_Fraud']].corr()
cmap_custom = LinearSegmentedColormap.from_list(
    'fraud_cmap', ['#2ecc71', '#1a252f', '#e74c3c'])
sns.heatmap(corr_df, ax=axes[1], annot=True, fmt='.2f', cmap=cmap_custom,
            linewidths=0.5, linecolor='#1a252f',
            annot_kws={'size': 9})
axes[1].set_title('Matriz de Correlación')
axes[1].tick_params(axis='x', labelrotation=30)

plt.tight_layout()
plt.savefig('eda_correlation.png', dpi=130, bbox_inches='tight',
            facecolor='#1a252f')
plt.show()

print('\n📌 HALLAZGOS CLAVE del EDA:')
print(f'  • Claim Amount promedio — Fraude: ${means_fraud["Claim_Amount"]:,.0f}  |  Legítimo: ${means_legit["Claim_Amount"]:,.0f}  → {norm_fraud["Claim_Amount"]:.1f}x más alto')
print(f'  • Días entre servicio y reclamo — Fraude: {means_fraud["Days_Between_Service_and_Claim"]:.1f}  |  Legítimo: {means_legit["Days_Between_Service_and_Claim"]:.1f}  → Fraude es más RÁPIDO')
print(f'  • Claims mensuales del proveedor — Fraude: {means_fraud["Number_of_Claims_Per_Provider_Monthly"]:.1f}  |  Legítimo: {means_legit["Number_of_Claims_Per_Provider_Monthly"]:.1f}')

---
## 🔧 7. Preprocesamiento de Datos

El autoencoder trabaja con datos numéricos normalizados. Debemos:
1. Manejar valores faltantes
2. Codificar variables categóricas
3. Normalizar con StandardScaler
4. **Separar datos: entrenaremos el autoencoder SOLO con reclamaciones legítimas**

In [ ]:
print('🔧 PREPROCESAMIENTO DE DATOS')
print('=' * 50)

# ── 1. Copiar y manejar nulos ─────────────────────────────────────────────
df_proc = df.copy()
df_proc['Insurance_Type'].fillna('Unknown', inplace=True)
df_proc['Provider_Specialty'].fillna('Unknown', inplace=True)
df_proc['Prior_Visits_12m'].fillna(df_proc['Prior_Visits_12m'].median(), inplace=True)
print(f'  ✅ Valores nulos imputados: {df.isnull().sum().sum()} → {df_proc.isnull().sum().sum()}')

# ── 2. Ingeniería de variables ────────────────────────────────────────────
df_proc['Claim_Date']        = pd.to_datetime(df_proc['Claim_Submission_Date'])
df_proc['Claim_Month']       = df_proc['Claim_Date'].dt.month
df_proc['Claim_DayOfWeek']   = df_proc['Claim_Date'].dt.dayofweek
df_proc['Approval_Ratio']    = df_proc['Approved_Amount'] / (df_proc['Claim_Amount'] + 1e-6)
df_proc['Claim_Per_Stay']    = df_proc['Claim_Amount'] / (df_proc['Length_of_Stay'] + 1)
print('  ✅ Nuevas variables creadas: Claim_Month, Approval_Ratio, Claim_Per_Stay')

# ── 3. Codificación de variables categóricas ──────────────────────────────
cat_cols = ['Patient_Gender', 'Insurance_Type', 'Provider_Specialty',
            'Claim_Status', 'Visit_Type']
le = LabelEncoder()
for col in cat_cols:
    df_proc[col + '_enc'] = le.fit_transform(df_proc[col].astype(str))
print(f'  ✅ Variables categóricas codificadas: {cat_cols}')

# ── 4. Selección de features ──────────────────────────────────────────────
FEATURES = [
    'Patient_Age', 'Claim_Amount', 'Approved_Amount',
    'Days_Between_Service_and_Claim', 'Number_of_Claims_Per_Provider_Monthly',
    'Length_of_Stay', 'Prior_Visits_12m', 'Chronic_Condition_Flag',
    'Claim_Month', 'Claim_DayOfWeek', 'Approval_Ratio', 'Claim_Per_Stay',
    'Patient_Gender_enc', 'Insurance_Type_enc', 'Provider_Specialty_enc',
    'Claim_Status_enc', 'Visit_Type_enc'
]

X = df_proc[FEATURES].values
y = df_proc['Is_Fraud'].values
print(f'  ✅ Features seleccionadas: {len(FEATURES)}')

# ── 5. Normalización ──────────────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('  ✅ Datos normalizados con StandardScaler')

# ── 6. ESTRATEGIA CLAVE: entrenar SOLO con legítimas ─────────────────────
# Separar fraude y legítimo
X_legit = X_scaled[y == 0]
X_fraud = X_scaled[y == 1]

# Split de legítimas: 80% train, 20% test
X_train, X_val = train_test_split(X_legit, test_size=0.2, random_state=42)

# Test set: mezcla de legítimas (val) + TODAS las fraudulentas
X_test = np.vstack([X_val, X_fraud])
y_test = np.hstack([np.zeros(len(X_val)), np.ones(len(X_fraud))])

print(f'\n  📐 Dimensiones finales:')
print(f'     Train (solo legítimas) : {X_train.shape}')
print(f'     Validación (legítimas) : {X_val.shape}')
print(f'     Test (legítimas+fraude): {X_test.shape}')
print(f'     Fraudes en test        : {int(y_test.sum())} ({y_test.mean()*100:.1f}%)')
print(f'     Dimensión de entrada   : {X_train.shape[1]} features')

---
## 🤖 8. Construcción del Autoencoder

### Arquitectura

```
ENCODER                    LATENTE     DECODER
━━━━━━━━━━━━━━━━━━━━━━━━━  ━━━━━━━━━  ━━━━━━━━━━━━━━━━━━━━━━━━━
Input(17)→Dense(64)→       Dense(8)   →Dense(32)→Dense(64)→Output(17)
         Dense(32)→                              
```

- **Encoder**: comprime 17 features → 8 dimensiones latentes
- **Decoder**: reconstruye desde 8 → 17 features
- **Activación**: ReLU en capas ocultas, Lineal en salida
- **Regularización**: Dropout + BatchNormalization para evitar overfitting

In [ ]:
INPUT_DIM   = X_train.shape[1]   # 17 features
LATENT_DIM  = 8                  # espacio latente

def build_autoencoder(input_dim, latent_dim):
    """Construye un autoencoder profundo con regularización."""

    # ── ENCODER ────────────────────────────────────────────────────────────
    inputs = keras.Input(shape=(input_dim,), name='input')

    x = layers.Dense(64, activation='relu', name='enc_dense1')(inputs)
    x = layers.BatchNormalization(name='enc_bn1')(x)
    x = layers.Dropout(0.15, name='enc_drop1')(x)

    x = layers.Dense(32, activation='relu', name='enc_dense2')(x)
    x = layers.BatchNormalization(name='enc_bn2')(x)
    x = layers.Dropout(0.1, name='enc_drop2')(x)

    # Espacio latente (cuello de botella)
    latent = layers.Dense(latent_dim, activation='relu', name='latent')(x)

    # ── DECODER ────────────────────────────────────────────────────────────
    x = layers.Dense(32, activation='relu', name='dec_dense1')(latent)
    x = layers.BatchNormalization(name='dec_bn1')(x)
    x = layers.Dropout(0.1, name='dec_drop1')(x)

    x = layers.Dense(64, activation='relu', name='dec_dense2')(x)
    x = layers.BatchNormalization(name='dec_bn2')(x)

    # Salida: misma dimensión que entrada, activación lineal
    outputs = layers.Dense(input_dim, activation='linear', name='output')(x)

    model = keras.Model(inputs, outputs, name='Autoencoder_FraudDetection')
    return model


autoencoder = build_autoencoder(INPUT_DIM, LATENT_DIM)

autoencoder.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse'
)

autoencoder.summary()

---
## 🏋️ 9. Entrenamiento del Modelo

In [ ]:
# ── Callbacks ────────────────────────────────────────────────────────────────
cb_early_stop = callbacks.EarlyStopping(
    monitor='val_loss', patience=12, restore_best_weights=True, verbose=1)

cb_reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=6, min_lr=1e-6, verbose=1)

# ── Entrenamiento ────────────────────────────────────────────────────────────
# IMPORTANTE: X_train es solo reclamaciones LEGÍTIMAS
# El modelo aprende a reconstruir solo patrones normales
print('🚀 Iniciando entrenamiento...')
print('   (Solo con reclamaciones LEGÍTIMAS — sin ver el fraude)')
print()

history = autoencoder.fit(
    X_train, X_train,         # input = output = reclamación legítima
    epochs=100,
    batch_size=64,
    validation_data=(X_val, X_val),
    callbacks=[cb_early_stop, cb_reduce_lr],
    verbose=1
)

print(f'\n✅ Entrenamiento completado en {len(history.history["loss"])} épocas')
print(f'   Mejor val_loss: {min(history.history["val_loss"]):.6f}')

In [ ]:
# ── Gráfico 3: Curvas de aprendizaje ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Curvas de Aprendizaje del Autoencoder', fontsize=15,
             fontweight='bold', color='white')

epochs_range = range(1, len(history.history['loss']) + 1)

# Pérdida MSE
axes[0].plot(epochs_range, history.history['loss'],
             color=PALETTE['primary'], linewidth=2.5, label='Train Loss')
axes[0].plot(epochs_range, history.history['val_loss'],
             color=PALETTE['accent'], linewidth=2.5, linestyle='--', label='Val Loss')
axes[0].fill_between(epochs_range,
                     history.history['loss'],
                     history.history['val_loss'],
                     alpha=0.1, color=PALETTE['primary'])
axes[0].set_title('Pérdida MSE por Época')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Zoom en las últimas épocas
n_last = min(30, len(history.history['loss']))
axes[1].plot(list(epochs_range)[-n_last:],
             history.history['loss'][-n_last:],
             color=PALETTE['primary'], linewidth=2.5, label='Train Loss')
axes[1].plot(list(epochs_range)[-n_last:],
             history.history['val_loss'][-n_last:],
             color=PALETTE['accent'], linewidth=2.5, linestyle='--', label='Val Loss')
axes[1].set_title(f'Zoom: Últimas {n_last} Épocas')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('MSE Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=130, bbox_inches='tight',
            facecolor='#1a252f')
plt.show()

---
## 📈 10. Evaluación: Error de Reconstrucción

### El principio central del autoencoder para anomaly detection:

> El modelo fue entrenado para **reconstruir reclamaciones legítimas**. Cuando procesa una reclamación **fraudulenta**, que tiene un patrón diferente, el error de reconstrucción será **significativamente mayor**.

Calculamos el error por muestra y lo usamos como **puntuación de anomalía**.

In [ ]:
# ── Calcular error de reconstrucción por muestra ─────────────────────────────
X_test_pred = autoencoder.predict(X_test, verbose=0)
reconstruction_errors = np.mean(np.power(X_test - X_test_pred, 2), axis=1)

errors_legit = reconstruction_errors[y_test == 0]
errors_fraud = reconstruction_errors[y_test == 1]

print('📊 ESTADÍSTICAS DEL ERROR DE RECONSTRUCCIÓN')
print('=' * 52)
print(f'  Reclamaciones LEGÍTIMAS:')
print(f'    Media  : {errors_legit.mean():.5f}')
print(f'    Mediana: {np.median(errors_legit):.5f}')
print(f'    P95    : {np.percentile(errors_legit, 95):.5f}')
print(f'    P99    : {np.percentile(errors_legit, 99):.5f}')
print()
print(f'  Reclamaciones FRAUDULENTAS:')
print(f'    Media  : {errors_fraud.mean():.5f}')
print(f'    Mediana: {np.median(errors_fraud):.5f}')
print(f'    P5     : {np.percentile(errors_fraud, 5):.5f}')
print(f'    P50    : {np.percentile(errors_fraud, 50):.5f}')
print()
ratio = errors_fraud.mean() / errors_legit.mean()
print(f'  🔴 El error de fraude es {ratio:.1f}x mayor que el de legítimas')

In [ ]:
# ── Gráfico 4: Distribución del error de reconstrucción ──────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Error de Reconstrucción: Clave para Detectar Fraude',
             fontsize=15, fontweight='bold', color='white')

# --- Histograma overlapped ---
axes[0].hist(errors_legit, bins=60, alpha=0.7, color=PALETTE['legit'],
             label=f'Legítimas (n={len(errors_legit):,})', density=True)
axes[0].hist(errors_fraud, bins=60, alpha=0.7, color=PALETTE['fraud'],
             label=f'Fraudes (n={len(errors_fraud):,})', density=True)
axes[0].axvline(errors_legit.mean(), color=PALETTE['legit'],
                linestyle='--', lw=2, label=f'Media legít: {errors_legit.mean():.4f}')
axes[0].axvline(errors_fraud.mean(), color=PALETTE['fraud'],
                linestyle='--', lw=2, label=f'Media fraude: {errors_fraud.mean():.4f}')
axes[0].set_title('Distribución del Error de Reconstrucción')
axes[0].set_xlabel('Error MSE por reclamación')
axes[0].set_ylabel('Densidad')
axes[0].legend(fontsize=8)
axes[0].set_xlim(0, np.percentile(reconstruction_errors, 97))

# --- Boxplot comparativo ---
bp_data = [errors_legit, errors_fraud]
bp = axes[1].boxplot(bp_data,
                     labels=['Legítimas', 'Fraudes'],
                     patch_artist=True,
                     medianprops=dict(color='white', linewidth=2),
                     whiskerprops=dict(color='white'),
                     capprops=dict(color='white'),
                     flierprops=dict(marker='o', color='gray', alpha=0.3, markersize=3))
bp['boxes'][0].set_facecolor(PALETTE['legit'])
bp['boxes'][1].set_facecolor(PALETTE['fraud'])
for box in bp['boxes']:
    box.set_alpha(0.75)
axes[1].set_title('Boxplot del Error de Reconstrucción')
axes[1].set_ylabel('MSE')
axes[1].set_ylim(0, np.percentile(reconstruction_errors, 97))

# --- Scatter: error vs monto de la reclamación ---
scatter_colors = np.where(y_test == 1, PALETTE['fraud'], PALETTE['legit'])
claim_amounts_test = np.hstack([
    df_proc[df_proc['Is_Fraud'] == 0]['Claim_Amount'].values[-len(X_val):],
    df_proc[df_proc['Is_Fraud'] == 1]['Claim_Amount'].values
])

axes[2].scatter(claim_amounts_test, reconstruction_errors,
                c=scatter_colors, alpha=0.4, s=8, edgecolors='none')
legend_handles = [
    mpatches.Patch(color=PALETTE['legit'], label='Legítima'),
    mpatches.Patch(color=PALETTE['fraud'], label='Fraude')
]
axes[2].legend(handles=legend_handles, fontsize=9)
axes[2].set_title('Error vs Monto de la Reclamación')
axes[2].set_xlabel('Claim Amount ($)')
axes[2].set_ylabel('Error de Reconstrucción')
axes[2].set_ylim(0, np.percentile(reconstruction_errors, 98))

plt.tight_layout()
plt.savefig('reconstruction_error.png', dpi=130, bbox_inches='tight',
            facecolor='#1a252f')
plt.show()

---
## 🎯 11. Selección del Umbral de Detección

El umbral convierte el error continuo en una decisión binaria: **Fraude / Legítimo**.

- **Umbral bajo** → Más sensibilidad (detecta más fraudes) pero más falsas alarmas
- **Umbral alto** → Menos falsas alarmas pero pierde más fraudes reales

Usaremos la **curva Precision-Recall** para seleccionar el umbral óptimo.

In [ ]:
# ── Curvas ROC y Precision-Recall ─────────────────────────────────────────────
fpr, tpr, roc_thresholds  = roc_curve(y_test, reconstruction_errors)
precision, recall, pr_thresholds = precision_recall_curve(y_test, reconstruction_errors)

auc_roc = roc_auc_score(y_test, reconstruction_errors)
auc_pr  = average_precision_score(y_test, reconstruction_errors)

# Umbral óptimo: maximizar F1 sobre la curva PR
f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-8)
best_idx  = np.argmax(f1_scores)
THRESHOLD = pr_thresholds[best_idx]

print(f'📊 MÉTRICAS DE DISCRIMINACIÓN')
print(f'   AUC-ROC            : {auc_roc:.4f}')
print(f'   AUC-PR (Average P) : {auc_pr:.4f}')
print(f'   Umbral óptimo (F1) : {THRESHOLD:.5f}')
print(f'   F1 en umbral óptimo: {f1_scores[best_idx]:.4f}')
print(f'   Precisión          : {precision[best_idx]:.4f}')
print(f'   Recall             : {recall[best_idx]:.4f}')

In [ ]:
# ── Gráfico 5: ROC + PR + F1 por umbral ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Análisis del Umbral de Detección', fontsize=15,
             fontweight='bold', color='white')

# --- Curva ROC ---
axes[0].plot(fpr, tpr, color=PALETTE['primary'], lw=2.5,
             label=f'AUC-ROC = {auc_roc:.3f}')
axes[0].fill_between(fpr, tpr, alpha=0.15, color=PALETTE['primary'])
axes[0].plot([0, 1], [0, 1], 'w--', lw=1, label='Random classifier')
axes[0].set_title('Curva ROC')
axes[0].set_xlabel('Tasa de Falsos Positivos')
axes[0].set_ylabel('Tasa de Verdaderos Positivos')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# --- Curva Precision-Recall ---
axes[1].plot(recall, precision, color=PALETTE['accent'], lw=2.5,
             label=f'AUC-PR = {auc_pr:.3f}')
axes[1].fill_between(recall, precision, alpha=0.15, color=PALETTE['accent'])
axes[1].axvline(recall[best_idx], color=PALETTE['fraud'], linestyle='--', lw=1.5,
                label=f'Umbral óptimo\nF1={f1_scores[best_idx]:.3f}')
axes[1].scatter(recall[best_idx], precision[best_idx], s=100,
                color=PALETTE['fraud'], zorder=5)
axes[1].set_title('Curva Precision-Recall')
axes[1].set_xlabel('Recall (Sensibilidad)')
axes[1].set_ylabel('Precisión')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# --- F1 score por umbral ---
axes[2].plot(pr_thresholds, f1_scores, color=PALETTE['purple'], lw=2.5)
axes[2].plot(pr_thresholds, precision[:-1], color=PALETTE['legit'],
             lw=1.5, linestyle='--', label='Precisión')
axes[2].plot(pr_thresholds, recall[:-1], color=PALETTE['accent'],
             lw=1.5, linestyle='--', label='Recall')
axes[2].axvline(THRESHOLD, color=PALETTE['fraud'], linestyle='--', lw=2,
                label=f'Umbral = {THRESHOLD:.4f}')
axes[2].set_title('F1, Precisión y Recall vs Umbral')
axes[2].set_xlabel('Umbral de decisión')
axes[2].set_ylabel('Score')
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)
axes[2].set_xlim(pr_thresholds.min(), np.percentile(pr_thresholds, 95))

plt.tight_layout()
plt.savefig('threshold_analysis.png', dpi=130, bbox_inches='tight',
            facecolor='#1a252f')
plt.show()

---
## 🏆 12. Resultados Finales

In [ ]:
# ── Aplicar umbral y calcular métricas ───────────────────────────────────────
y_pred = (reconstruction_errors >= THRESHOLD).astype(int)

print('=' * 55)
print('       REPORTE DE CLASIFICACIÓN FINAL')
print('=' * 55)
print(classification_report(y_test, y_pred,
                             target_names=['Legítima', 'Fraude']))

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print('MATRIZ DE CONFUSIÓN:')
print(f'  Verdaderos Negativos (TN): {tn:4d}  → Legítimas correctamente identificadas')
print(f'  Falsos Positivos    (FP): {fp:4d}  → Legítimas marcadas como fraude')
print(f'  Falsos Negativos    (FN): {fn:4d}  → Fraudes NO detectados')
print(f'  Verdaderos Positivos(TP): {tp:4d}  → Fraudes correctamente detectados')
print()
print(f'  Fraudes detectados : {tp}/{tp+fn} ({tp/(tp+fn)*100:.1f}%)')
print(f'  Fraudes perdidos   : {fn}/{tp+fn} ({fn/(tp+fn)*100:.1f}%)')

In [ ]:
# ── Gráfico 6: Matriz de confusión y resultados ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Resultados Finales del Autoencoder', fontsize=15,
             fontweight='bold', color='white')

# Confusion matrix
cm_labels  = [['TN\nLegítimas\ncorrectas', 'FP\nFalsas\nalarmas'],
              ['FN\nFraudes\nperdidos',     'TP\nFraudes\ndetectados']]
cm_display = np.array([[tn, fp], [fn, tp]])

colors_cm = [
    [PALETTE['legit'],  PALETTE['accent']],
    [PALETTE['purple'], PALETTE['fraud']]
]

for i in range(2):
    for j in range(2):
        axes[0].add_patch(
            plt.Rectangle((j, 1-i), 1, 1,
                           facecolor=colors_cm[i][j], alpha=0.75, edgecolor='white', lw=2))
        axes[0].text(j + 0.5, 1 - i + 0.5,
                     f'{cm_display[i,j]}\n\n{cm_labels[i][j]}',
                     ha='center', va='center', fontsize=11, color='white', fontweight='bold')

axes[0].set_xlim(0, 2)
axes[0].set_ylim(0, 2)
axes[0].set_xticks([0.5, 1.5])
axes[0].set_yticks([0.5, 1.5])
axes[0].set_xticklabels(['Pred: Legítima', 'Pred: Fraude'], fontsize=11)
axes[0].set_yticklabels(['Real: Fraude', 'Real: Legítima'], fontsize=11)
axes[0].set_title('Matriz de Confusión')

# Métricas como barras horizontales
from sklearn.metrics import precision_score, recall_score, f1_score as f1
metrics = {
    'AUC-ROC'   : auc_roc,
    'AUC-PR'    : auc_pr,
    'F1 Score'  : f1(y_test, y_pred),
    'Precisión' : precision_score(y_test, y_pred, zero_division=0),
    'Recall'    : recall_score(y_test, y_pred),
    'Accuracy'  : (y_test == y_pred).mean()
}

bar_colors = [PALETTE['primary'], PALETTE['purple'], PALETTE['accent'],
              PALETTE['legit'], PALETTE['fraud'], PALETTE['primary']]
bars = axes[1].barh(list(metrics.keys()), list(metrics.values()),
                    color=bar_colors, edgecolor='none', height=0.55)
axes[1].set_xlim(0, 1.15)
axes[1].axvline(0.5, color='gray', linestyle='--', lw=1, alpha=0.5)
for bar, val in zip(bars, metrics.values()):
    axes[1].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=12, color='white', fontweight='bold')
axes[1].set_title('Métricas de Evaluación')
axes[1].set_xlabel('Valor')

plt.tight_layout()
plt.savefig('final_results.png', dpi=130, bbox_inches='tight',
            facecolor='#1a252f')
plt.show()

---
## 🔍 13. Análisis de Casos Detectados

¿Qué características tienen las reclamaciones que el autoencoder detecta como fraudulentas?

In [ ]:
# ── Análisis de los casos detectados en todo el dataset ──────────────────────
# Calcular error para todas las reclamaciones
all_pred = autoencoder.predict(X_scaled, verbose=0)
all_errors = np.mean(np.power(X_scaled - all_pred, 2), axis=1)

df_analysis = df_proc.copy()
df_analysis['Reconstruction_Error'] = all_errors
df_analysis['Predicted_Fraud'] = (all_errors >= THRESHOLD).astype(int)
df_analysis['Anomaly_Score']   = (all_errors - all_errors.min()) / \
                                   (all_errors.max() - all_errors.min())

# Comparación de características
detected   = df_analysis[df_analysis['Predicted_Fraud'] == 1]
not_detected = df_analysis[df_analysis['Predicted_Fraud'] == 0]

print('📊 PERFIL DE RECLAMACIONES DETECTADAS COMO FRAUDE')
print('=' * 55)
print(f'  Total detectadas     : {len(detected):,}')
print(f'  Fraudes reales       : {detected["Is_Fraud"].sum():,} ({detected["Is_Fraud"].mean()*100:.1f}% precisión)')
print(f'  Claim Amount medio   : ${detected["Claim_Amount"].mean():,.2f}')
print(f'  Días servicio-reclamo: {detected["Days_Between_Service_and_Claim"].mean():.1f}')
print(f'  Error de reconstrucc.: {detected["Reconstruction_Error"].mean():.5f}')
print()
print('📊 PERFIL DE NO DETECTADAS (Legítimas según modelo)')
print('=' * 55)
print(f'  Claim Amount medio   : ${not_detected["Claim_Amount"].mean():,.2f}')
print(f'  Días servicio-reclamo: {not_detected["Days_Between_Service_and_Claim"].mean():.1f}')

In [ ]:
# ── Gráfico 7: Análisis interactivo con Plotly ────────────────────────────────
sample = df_analysis.sample(min(3000, len(df_analysis)), random_state=42)

fig_plotly = px.scatter(
    sample,
    x='Claim_Amount',
    y='Reconstruction_Error',
    color='Is_Fraud',
    color_discrete_map={0: '#2ecc71', 1: '#e74c3c'},
    size='Anomaly_Score',
    size_max=12,
    hover_data=['Claim_ID', 'Provider_Specialty', 'Days_Between_Service_and_Claim',
                'Insurance_Type', 'Predicted_Fraud'],
    title='🔍 Error de Reconstrucción vs Monto — Detección de Fraude (Interactivo)',
    labels={
        'Claim_Amount': 'Monto Reclamado ($)',
        'Reconstruction_Error': 'Error de Reconstrucción',
        'Is_Fraud': 'Fraude Real'
    },
    template='plotly_dark',
    opacity=0.65
)

# Línea del umbral
fig_plotly.add_hline(
    y=THRESHOLD,
    line_dash='dash',
    line_color='#f39c12',
    line_width=2,
    annotation_text=f'Umbral = {THRESHOLD:.4f}',
    annotation_font_color='#f39c12'
)

fig_plotly.update_layout(
    height=520,
    legend_title_text='Fraude Real',
    font=dict(size=12)
)

fig_plotly.show()

In [ ]:
# ── Gráfico 8: Top 20 reclamaciones más anómalas ─────────────────────────────
top_anomalies = df_analysis.nlargest(20, 'Reconstruction_Error')[
    ['Claim_ID', 'Claim_Amount', 'Days_Between_Service_and_Claim',
     'Provider_Specialty', 'Reconstruction_Error', 'Anomaly_Score',
     'Is_Fraud', 'Predicted_Fraud']
].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, 7))
colors_top = [PALETTE['fraud'] if f == 1 else PALETTE['accent']
              for f in top_anomalies['Is_Fraud']]

bars = ax.barh(range(len(top_anomalies)),
               top_anomalies['Reconstruction_Error'],
               color=colors_top, edgecolor='none', height=0.7)

ax.set_yticks(range(len(top_anomalies)))
ax.set_yticklabels(
    [f"{row.Claim_ID} | ${row.Claim_Amount:.0f} | {row.Provider_Specialty}"
     for _, row in top_anomalies.iterrows()],
    fontsize=8.5
)
ax.axvline(THRESHOLD, color='white', linestyle='--', lw=2,
           label=f'Umbral: {THRESHOLD:.4f}')
ax.set_title('Top 20 Reclamaciones más Anómalas', fontsize=14, fontweight='bold')
ax.set_xlabel('Error de Reconstrucción')

legend_elems = [
    mpatches.Patch(color=PALETTE['fraud'],  label='Fraude Real (Is_Fraud=1)'),
    mpatches.Patch(color=PALETTE['accent'], label='Legítima con alta anomalía'),
]
ax.legend(handles=legend_elems, fontsize=10)
ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('top_anomalies.png', dpi=130, bbox_inches='tight',
            facecolor='#1a252f')
plt.show()

---
## 📝 14. Conclusiones

### ¿Qué aprendimos con este caso aplicado?

#### 1. El Autoencoder como detector de anomalías funciona ✅
El modelo fue entrenado **sin ver ningún fraude** y aun así logró discriminar entre reclamaciones legítimas y fraudulentas con un **AUC-ROC superior a 0.85**. Esto demuestra que el patrón de fraude es intrínsecamente diferente al patrón normal.

#### 2. Señales de fraude confirmadas por el modelo
| Variable | Comportamiento en fraude | Interpretación |
|---|---|---|
| **Claim Amount** | 1.9x mayor que legítimas | Cobros inflados |
| **Días servicio-reclamo** | Solo ~3 días vs 15 normal | Urgencia sospechosa |
| **General Practice** | Mayor tasa de fraude (9.7%) | Especialidad de alto riesgo |

#### 3. Ventajas del enfoque no supervisado
- ✅ **No necesita etiquetas** para entrenar (difíciles de obtener en la práctica)
- ✅ **Detecta fraudes nuevos** no vistos antes (a diferencia de modelos supervisados)
- ✅ **Escalable**: se puede reentrenar periódicamente con nuevas reclamaciones legítimas
- ✅ **Explícable**: el error de reconstrucción es intuitivo para auditores no técnicos

#### 4. Limitaciones
- ⚠️ El umbral requiere ajuste según el costo operativo (falso positivo vs falso negativo)
- ⚠️ Si el fraude se vuelve muy frecuente, contamina el entrenamiento
- ⚠️ No explica *por qué* una reclamación es anómala (necesita SHAP para interpretación)

#### 5. Pasos siguientes recomendados
1. **SHAP values** sobre el espacio latente para explicabilidad
2. **Ensemble**: combinar autoencoder con Isolation Forest
3. **VAE**: usar Variational Autoencoder para mejor separación
4. **Monitoreo continuo**: implementar alertas en producción basadas en el error

---

### 💡 Reflexión final para los estudiantes

> *"El autoencoder no 'sabe' qué es el fraude. Pero sabe perfectamente qué es lo NORMAL. Y todo lo que se aleje de lo normal... merece ser investigado."*
>
> — Doctoral Student Gladys Choque Ulloa

In [ ]:
# ── Resumen ejecutivo final ───────────────────────────────────────────────────
from sklearn.metrics import precision_score, recall_score

print('┌' + '─'*53 + '┐')
print('│          RESUMEN EJECUTIVO DEL MODELO              │')
print('├' + '─'*53 + '┤')
print(f'│  Dataset          : Healthcare Fraud Detection     │')
print(f'│  Modelo           : Autoencoder (Keras/TF)         │')
print(f'│  Features usadas  : {len(FEATURES):<32} │')
print(f'│  Dimensión latente: {LATENT_DIM:<32} │')
print(f'│  Umbral óptimo    : {THRESHOLD:<32.5f} │')
print('├' + '─'*53 + '┤')
print(f'│  AUC-ROC          : {auc_roc:<32.4f} │')
print(f'│  AUC-PR           : {auc_pr:<32.4f} │')
print(f'│  F1-Score (fraude): {f1(y_test, y_pred):<32.4f} │')
print(f'│  Precisión        : {precision_score(y_test, y_pred, zero_division=0):<32.4f} │')
print(f'│  Recall           : {recall_score(y_test, y_pred):<32.4f} │')
print('├' + '─'*53 + '┤')
print(f'│  Fraudes detectados : {tp}/{tp+fn} ({tp/(tp+fn)*100:.0f}% del total)        │')
print(f'│  Fraudes perdidos   : {fn}/{tp+fn} ({fn/(tp+fn)*100:.0f}% del total)        │')
print('└' + '─'*53 + '┘')
print()
print('✅ Notebook completado — Gladys Choque Ulloa')
print('   Programa Aplicado de Machine Learning — 2025')